In [ ]:
# Example 1: bfloat16 dtype basics.
import numpy as np
import ml_dtypes

bf16 = ml_dtypes.bfloat16
arr = np.array([1.0, 2.5, -3.75, 0.125], dtype=bf16)
print(f"dtype: {arr.dtype}")
print(f"values: {arr}")
print(f"itemsize: {arr.itemsize} bytes")

# Conversion round-trip via float32.
as_f32 = arr.astype(np.float32)
back_to_bf16 = as_f32.astype(bf16)
print(f"as float32: {as_f32}")
print(f"back to bf16: {back_to_bf16}")

assert arr.dtype == bf16
assert arr.itemsize == 2
assert np.array_equal(arr, back_to_bf16)

In [ ]:
# Example 2: float8 dtype family (E4M3FN, E5M2).
import numpy as np
import ml_dtypes

e4m3 = ml_dtypes.float8_e4m3fn
e5m2 = ml_dtypes.float8_e5m2

vals = np.array([0.0, 0.5, 1.0, 2.0, -1.5], dtype=np.float32)
arr_e4m3 = vals.astype(e4m3)
arr_e5m2 = vals.astype(e5m2)

print(f"e4m3fn: {arr_e4m3}  itemsize={arr_e4m3.itemsize}")
print(f"e5m2:   {arr_e5m2}  itemsize={arr_e5m2.itemsize}")

assert arr_e4m3.dtype == e4m3
assert arr_e5m2.dtype == e5m2
assert arr_e4m3.itemsize == 1
assert arr_e5m2.itemsize == 1

In [ ]:
# Example 3: finfo for ML dtypes — exercises numeric metadata.
import ml_dtypes
import numpy as np

for dt in (ml_dtypes.bfloat16, ml_dtypes.float8_e4m3fn, ml_dtypes.float8_e5m2):
    info = ml_dtypes.finfo(dt)
    print(f"{dt.__name__}: bits={info.bits} eps={float(info.eps):.4g} "
          f"max={float(info.max):.4g} min={float(info.min):.4g}")
    assert info.bits in (8, 16)
    assert info.max > 0
    assert info.min < 0

# bfloat16 has the same exponent range as float32 (8-bit exponent).
bf16_info = ml_dtypes.finfo(ml_dtypes.bfloat16)
f32_info = np.finfo(np.float32)
assert bf16_info.maxexp == f32_info.maxexp

In [ ]:
# Example 4: int4 / uint4 sub-byte integer dtypes.
import numpy as np
import ml_dtypes

int4 = ml_dtypes.int4
uint4 = ml_dtypes.uint4

i = np.array([-8, -1, 0, 1, 7], dtype=int4)
u = np.array([0, 1, 7, 15], dtype=uint4)
print(f"int4 values:  {i}  dtype={i.dtype}")
print(f"uint4 values: {u}  dtype={u.dtype}")

iinfo_i = ml_dtypes.iinfo(int4)
iinfo_u = ml_dtypes.iinfo(uint4)
print(f"int4 range:  [{iinfo_i.min}, {iinfo_i.max}]")
print(f"uint4 range: [{iinfo_u.min}, {iinfo_u.max}]")

assert iinfo_i.min == -8 and iinfo_i.max == 7
assert iinfo_u.min == 0 and iinfo_u.max == 15

In [ ]:
# Example 5: arithmetic + numpy interop on bfloat16.
import numpy as np
import ml_dtypes

bf16 = ml_dtypes.bfloat16
a = np.array([1.0, 2.0, 3.0, 4.0], dtype=bf16)
b = np.array([0.5, 0.5, 0.5, 0.5], dtype=bf16)

c = a + b
d = a * b
dot = np.dot(a.astype(np.float32), b.astype(np.float32))

print(f"a + b = {c}")
print(f"a * b = {d}")
print(f"dot(a, b) [via f32] = {dot}")

assert c.dtype == bf16
assert d.dtype == bf16
assert np.allclose(c.astype(np.float32), [1.5, 2.5, 3.5, 4.5])
assert np.allclose(d.astype(np.float32), [0.5, 1.0, 1.5, 2.0])
assert float(dot) == 5.0